In [81]:
import pandas as pd
### 所需要的所有文件：
# PLM的生命周期全表
# 物料基本信息_MARA，需要创建时间
# 近一年整机生产订单
# 近一年整机采购订单
# 所有整机的制造BOM

In [82]:
productline_list = ['油烟机产品线', '烹饪厨电产品线', '洗碗机产品线', '净热产品线','冰储产品线']

### 处理PLM导出的产品生命周期状态全表（保留13位物料号，国内，5大产品线）

In [83]:
df_plm = pd.read_excel(r"D:\000物料报表\物料运维报告\产品生命周期状态全表.xlsx")
len(df_plm)

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


9136

In [85]:
df_plm['物料号'] = df_plm['物料号'].astype(str).str[:13]
df_plm = df_plm[df_plm['物料号'].str.len() == 13]
df_plm = df_plm[df_plm['国内/海外'] == '国内']
df_plm = df_plm[df_plm['产品线'].isin(productline_list)]
len(df_plm)
# df_plm.columns

6792

### 整机概览

In [68]:
# 创建初始DataFrame
df1 = pd.DataFrame()
df1['产品线'] = productline_list
# 定义状态列
status_columns = ['开发','样机','小批量','项目阶段小计','量产','退市预警',
                  '停止销售','在产阶段小计','停止生产','停止发货','停止服务',
                  '作废','淘汰阶段小计']
# 初始化状态列为0
df1[status_columns] = 0
# 遍历每条产品线
for index, row in df1.iterrows():
    product_line = row['产品线']
    # 遍历每个状态列
    for col in status_columns:
        # 统计符合条件的唯一物料号数量
        count = df_plm[(df_plm['产品线'] == product_line) & 
                      (df_plm['产品状态'] == col)]['物料号'].nunique()
        # 使用.loc更新DataFrame的值
        df1.loc[index, col] = count
# 计算各阶段小计
for index in df1.index:
    # 项目阶段小计 = 开发 + 样机 + 小批量
    df1.loc[index, '项目阶段小计'] = df1.loc[index, ['开发', '样机', '小批量']].sum()
    # 在产阶段小计 = 量产 + 退市预警 + 停止销售
    df1.loc[index, '在产阶段小计'] = df1.loc[index, ['量产', '退市预警', '停止销售']].sum()
    # 淘汰阶段小计 = 停止生产 + 停止发货 + 停止服务 + 作废
    df1.loc[index, '淘汰阶段小计'] = df1.loc[index, ['停止生产', '停止发货', '停止服务', '作废']].sum()
# 计算各列的合计值（排除'产品线'列）
total = df1.drop('产品线', axis=1).sum()
# 创建合计行
total_row = pd.DataFrame([total], index=['合计'])
total_row['产品线'] = '国内合计'  # 添加产品线标签
# 调整列顺序，保持与原DataFrame一致
total_row = total_row[df1.columns]
# 将合计行添加到原DataFrame末尾
df1_with_total = pd.concat([df1, total_row], ignore_index=True)
df1_with_total


,产品线,开发,样机,小批量,项目阶段小计,量产,退市预警,停止销售,在产阶段小计,停止生产,停止发货,停止服务,作废,淘汰阶段小计
0,油烟机产品线,3,12,38,53,262,13,96,371,21,438,97,72,628
1,烹饪厨电产品线,23,19,39,81,440,226,191,857,26,1713,0,10,1749
2,洗碗机产品线,4,6,18,28,105,19,70,194,2,82,0,15,99
3,净热产品线,0,4,6,10,113,33,24,170,15,799,0,15,829
4,冰储产品线,0,11,0,11,60,4,10,74,4,91,0,3,98
5,国内合计,30,52,101,183,980,295,391,1666,68,3123,97,115,3403


### 各产品线整机存活情况

#### 将创建日期匹配进PLM生命周期全表，并添加生命周期对应阶段列

In [ ]:
df_mara = pd.read_excel(r"D:\000物料报表\物料运维报告\物料基本信息_MARA.XLSX")
df_mara['物料编码'] = df_mara['物料编码'].astype(str).str[:13]
df_mara['制造方式'] = df_mara['制造方式'].astype(str).str[:2]
len(df_mara)


219061

In [70]:
df_plm_mara_left = pd.merge(df_plm, df_mara[['物料编码','创建日期']], left_on='物料号', right_on='物料编码', how='left')
len(df_plm_mara_left)


6792

In [71]:
df_plm_mara_left['生命周期对应阶段'] = df_plm_mara_left['产品状态'].apply(lambda x: '项目阶段' if x in ('开发','样机','小批量') else '在产阶段' if x in ('量产','退市预警','停止销售') else '淘汰阶段' if x in ('停止生产','停止发货','停止服务','作废') else '其他')
df_plm_mara_left['生命周期对应阶段'].value_counts()


生命周期对应阶段
淘汰阶段    3779
在产阶段    2824
项目阶段     189
Name: count, dtype: int64

#### 构建各产品线整机存活情况

In [72]:
df2 = pd.DataFrame()
df2['产品线'] = productline_list
status_columns = ['产品线','2022年新增','2022年新增未淘汰','2022年存活率','2023年新增','2023年新增未淘汰','2023年存活率','2024年新增','2024年新增未淘汰','2024年存活率','2025年新增','2025年新增未淘汰','2025年存活率','型号数','未淘汰']
for index,row in df2.iterrows():
    product_line = row['产品线']
    df2.loc[index,'2022年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2022)]['物料号'].nunique()
    df2.loc[index,'2022年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2022) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'2023年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2023)]['物料号'].nunique()
    df2.loc[index,'2023年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2023) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'2024年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2024)]['物料号'].nunique()
    df2.loc[index,'2024年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2024) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()    
    df2.loc[index,'2025年新增'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2025)]['物料号'].nunique()
    df2.loc[index,'2025年新增未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['创建日期'].dt.year == 2025) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()
    df2.loc[index,'型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line)]['物料号'].nunique()
    df2.loc[index,'未淘汰'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'].isin(['项目阶段','在产阶段']))]['物料号'].nunique()  
df2
# 计算各列的合计值（排除'产品线'列）
total = df2.drop('产品线', axis=1).sum()
# 创建合计行
total_row = pd.DataFrame([total], index=['合计'])
total_row['产品线'] = '国内合计'  # 添加产品线标签
# 调整列顺序，保持与原DataFrame一致
total_row = total_row[df2.columns]
total_row
# 将合计行添加到原DataFrame末尾
df2_with_total = pd.concat([df2, total_row], ignore_index=True)
df2_with_total
# 计算2022年、2023年、2024年的存活率
df2_with_total['2022年存活率'] = df2_with_total['2022年新增未淘汰'] / df2_with_total['2022年新增']
df2_with_total['2023年存活率'] = df2_with_total['2023年新增未淘汰'] / df2_with_total['2023年新增']
df2_with_total['2024年存活率'] = df2_with_total['2024年新增未淘汰'] / df2_with_total['2024年新增']
df2_with_total['2025年存活率'] = df2_with_total['2025年新增未淘汰'] / df2_with_total['2025年新增']
df2_with_total = df2_with_total[status_columns]
df2_with_total



,产品线,2022年新增,2022年新增未淘汰,2022年存活率,2023年新增,2023年新增未淘汰,2023年存活率,2024年新增,2024年新增未淘汰,2024年存活率,2025年新增,2025年新增未淘汰,2025年存活率,型号数,未淘汰
0,油烟机产品线,75.0,65.0,0.866667,46.0,43.0,0.934783,71.0,71.0,1.000000,98.0,98.0,1.0,1052.0,424.0
1,烹饪厨电产品线,135.0,116.0,0.859259,151.0,150.0,0.993377,127.0,123.0,0.968504,149.0,149.0,1.0,2687.0,938.0
2,洗碗机产品线,26.0,18.0,0.692308,59.0,52.0,0.881356,56.0,48.0,0.857143,39.0,39.0,1.0,321.0,222.0
3,净热产品线,27.0,21.0,0.777778,36.0,36.0,1.000000,32.0,32.0,1.000000,14.0,14.0,1.0,1009.0,180.0
4,冰储产品线,9.0,6.0,0.666667,10.0,9.0,0.900000,26.0,26.0,1.000000,12.0,12.0,1.0,183.0,85.0
5,国内合计,272.0,226.0,0.830882,302.0,290.0,0.960265,312.0,300.0,0.961538,312.0,312.0,1.0,5252.0,1849.0


### 在产阶段，整机的生产,采购订单状况

In [73]:
df_maked = pd.read_excel(r"D:\000物料报表\物料运维报告\近一年整机生产订单.XLSX")
df_maked['物料编号'] = df_maked['物料编号'].astype(str).str[:13]
df_buyer = pd.read_excel(r"D:\000物料报表\物料运维报告\近一年整机采购订单.XLSX")
df_buyer['物料编码'] = df_buyer['物料编码'].astype(str).str[:13]
len(df_maked),len(df_buyer)


(43962, 29922)

#### 选出近一年生产过的物料编码，然后再新建列近一年是否有生产订单，来标记

In [76]:
maked_product = set(df_maked['物料编号'])
buyer_product = set(df_buyer['物料编码'])
df_plm_mara_left['近一年是否有生产订单'] = df_plm_mara_left['物料号'].apply(lambda x: '是' if x in maked_product else '否')
df_plm_mara_left['近一年是否有采购订单'] = df_plm_mara_left['物料号'].apply(lambda x: '是' if x in buyer_product else '否')
df_plm_mara_left['近一年是否有采购订单'].value_counts(),df_plm_mara_left['近一年是否有生产订单'].value_counts()


(近一年是否有采购订单
 否    5453
 是    1339
 Name: count, dtype: int64,
 近一年是否有生产订单
 否    4871
 是    1921
 Name: count, dtype: int64)

#### 构造整机在在产阶段的生产订单情况

In [ ]:

df3 = pd.DataFrame()
status_columns = ['产品线','在产阶段的型号数','有生产订单的型号数','有采购订单的型号数','有采购&生产订单的型号数','有排单的型号数占比']
df3['产品线'] = productline_list
for index,row in df3.iterrows():
    product_line = row['产品线']
    df3.loc[index,'在产阶段的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段')]['物料号'].nunique()
    df3.loc[index,'有生产订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') & (df_plm_mara_left['近一年是否有生产订单'] == '是')]['物料号'].nunique()
    df3.loc[index,'有采购订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') & (df_plm_mara_left['近一年是否有采购订单'] == '是')]['物料号'].nunique()
    df3.loc[index,'有采购生产订单的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '在产阶段') &  ((df_plm_mara_left['近一年是否有采购订单'] == '是') | (df_plm_mara_left['近一年是否有生产订单'] == '是'))]['物料号'].nunique()
df3

,产品线,在产阶段的型号数,有生产订单的型号数,有采购订单的型号数,有采购/生产订单的型号数
0,油烟机产品线,371.0,224.0,82.0,241.0
1,烹饪厨电产品线,857.0,484.0,187.0,506.0
2,洗碗机产品线,194.0,157.0,71.0,168.0
3,净热产品线,170.0,122.0,79.0,140.0
4,冰储产品线,74.0,46.0,48.0,72.0


### 淘汰阶段的整机统计

#### 读取产品的库存数据，并识别出哪些是有库存的整机，然后再新建列是否有库存，来标记

In [46]:
df_stock = pd.read_excel(r"D:\000物料报表\物料运维报告\整机仓库库存.XLSX")
df_stock['物料编码'] = df_stock['物料编码'].astype(str).str[:13]
len(df_stock['物料编码'].unique())


2786

In [ ]:
stored_product = set(df_stock['物料编码'])
df_plm_mara_left['是否有库存'] = df_plm_mara_left['物料编码'].apply(lambda x: '是' if x in stored_product else '无')
df_plm_mara_left['是否有库存'].value_counts()


是否有库存
是    3691
无    3101
Name: count, dtype: int64

#### 构建淘汰阶段的库存情况统计

In [ ]:

df4 = pd.DataFrame()
status_columns = ['产品线','淘汰阶段的型号数','有库存的型号数']
df4['产品线'] = productline_list
for index,row in df4.iterrows():
    product_line = row['产品线']
    df4.loc[index,'淘汰阶段的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '淘汰阶段')]['物料号'].nunique()
    df4.loc[index,'有库存的型号数'] = df_plm_mara_left[(df_plm_mara_left['产品线'] == product_line) & (df_plm_mara_left['生命周期对应阶段'] == '淘汰阶段') & (df_plm_mara_left['是否有库存'] == '是')]['物料号'].nunique()
df4


,产品线,淘汰阶段的型号数,有库存的型号数
0,油烟机产品线,628.0,241.0
1,烹饪厨电产品线,1749.0,566.0
2,洗碗机产品线,99.0,69.0
3,净热产品线,829.0,234.0
4,冰储产品线,98.0,43.0


### 零件概览

In [93]:
# 10开头是整机、11：零部件、12：电子元器件、13：紧固密封件、14：板材、15：其他物料
for index,row in df_mara.iterrows():
    if row['物料编码'][:2] == '10':
        df_mara.loc[index,'物料类型'] = '整机'
    elif row['物料编码'][:2] == '11':
        df_mara.loc[index,'物料类型'] = '零部件'
    elif row['物料编码'][:2] == '12':
        df_mara.loc[index,'物料类型'] = '电子元器件'
    elif row['物料编码'][:2] == '13':
        df_mara.loc[index,'物料类型'] = '紧固密封件'
    elif row['物料编码'][:2] == '14':
        df_mara.loc[index,'物料类型'] = '板材'
    elif row['物料编码'][:2] == '15':
        df_mara.loc[index,'物料类型'] = '其他物料'


for index,row in df_mara.iterrows():
    if row['物料编码'][:4] in ['1101']:
        df_mara.loc[index,'零部件所用产品线'] = '油烟机产品线'
    if row['物料编码'][:4] in ['1102','1105','1106','1107','1109','1110','1111','1116']:
        df_mara.loc[index,'零部件所用产品线'] = '烹饪厨电产品线'
    if row['物料编码'][:4] in ['1108','1118','1124','1126']:
        df_mara.loc[index,'零部件所用产品线'] = '洗碗机产品线'
    if row['物料编码'][:4] in ['1104','1113','1114']:
        df_mara.loc[index,'零部件所用产品线'] = '净热产品线'
    if row['物料编码'][:4] in ['1103','1119']:
        df_mara.loc[index,'零部件所用产品线'] = '冰储产品线'
    if row['物料编码'][:2] in ['12','13','14','15']:
        df_mara.loc[index,'零部件所用产品线'] = 'all'
df_mara['物料类型'].value_counts(),df_mara['零部件所用产品线'].value_counts()



(物料类型
 零部件      131534
 其他物料      65925
 电子元器件      9155
 整机         7032
 板材         4257
 紧固密封件       487
 Name: count, dtype: int64,
 零部件所用产品线
 all        79824
 烹饪厨电产品线    47808
 油烟机产品线     36799
 净热产品线      18914
 洗碗机产品线     15575
 冰储产品线       7781
 Name: count, dtype: int64)

In [89]:
df_mara.columns

Index(['物料编码', '创建日期', '是否冻结', '是否关键件', '是否服务配件', '是否服务专用配件', '产品品类',
       '是否服务虚拟件', '国内/海外', '产品线', '产品线描述', '物料描述', '物料类型'],
      dtype='object')

In [ ]:
from numpy import isin


df5 = pd.DataFrame()
df5['物料类型'] = ['零部件','零部件','零部件','零部件','零部件','电子元器件','紧固密封件','板材','其他物料']
df5['零部件所用产品线'] = ['油烟机产品线','烹饪厨电产品线','洗碗机产品线','净热产品线','冰储产品线','all','all','all','all']
for index,row in df5.iterrows():
    prodcut_type = row['物料类型']
    prodcut_type_line = row['零部件所用产品线']
    df5.loc[index,'自制数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')&(df_mara['制造方式']=='10')]['物料编码'].unique()
    df5.loc[index,'外购&外协'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')&(df_mara['制造方式']!='10')]['物料编码'].unique()
    df5.loc[index,'非冻结总计数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] != 'X')]['物料编码'].unique()
    df5.loc[index,'冻结数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & (df_mara['是否冻结'] == 'X')]['物料编码'].unique()
    df5.loc[index,'冻结&采购冻结数量'] = df_mara[(df_mara['物料类型']==prodcut_type) & (df_mara['零部件所用产品线'] == prodcut_type_line) & ((df_mara['是否冻结'] == 'X')|(df_mara['跨工厂物料状态'].isin(['04','01','05'])))]['物料编码'].unique()








,物料类型,零部件所用产品线
0,零部件,油烟机产品线
1,零部件,烹饪厨电产品线
2,零部件,洗碗机产品线
3,零部件,净热产品线
4,零部件,冰储产品线
5,电子元器件,all
6,紧固密封件,all
7,板材,all
8,其他物料,all
